### This video covers the implementation of:
 Neural Language Model with FF Neural Network

 
![](https://drive.google.com/uc?export=view&id=1cJvHX2Tq3V4YqtOMOA0OfWVtalYSuOhz)



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
from numpy import array

In [ ]:
# this function is defined to read Hamshahri corpus
def read_hamshahri_corpus(path):

  count = 0
  res = []
  corpus = ''

  with open(path, 'r') as ptr:
    for line in ptr:
      res.append(line)


  for i in range(0, len(res)):
    
    if count>500:
      break

    if res[i].startswith(".DID"):
      count += 1
      continue

    if res[i].startswith(".Date"):
      continue

    if res[i].startswith(".Cat"):
      continue

    corpus = corpus+ ' ' + res[i].strip()

  return corpus


In [ ]:
import string


# Returns an array of words that occurred less than N time in given corpus
def get_rare_words(corpus, N):
  
  tokens = corpus.split()
  
  dict = {}
  
  for word in tokens:
    
    if word in dict:
      dict[word]+=1
    
    else:
      dict[word]=1

  rare_words = []

  for key, value in dict.items():

    if value <= N:
      rare_words.append(key)

  print('count of rare words: ')
  print(len(rare_words))

  return rare_words


# remove punctuation marks, non-alphabetic tokens, and lowercase the tokens
def clean_sentence(sentence):

  tokens = sentence.split()

  translation_table = str.maketrans('', '', "\"#$%&'()*+,-/:;<=>?@[\]^_`{|}~،؟!❊#$٪^&*)(ـ+=-؛:{}")

  tokens = [w.translate(translation_table) for w in tokens]

  tokens = [word for word in tokens if word.isalpha()]

  tokens = [word for word in tokens if word not in rare_words]
  
  return tokens




In [ ]:

import pickle

# Read dataset, extract rare-words, split the corpus into sentences, clean each sentence and store the
# cleaned sentences with pickle

USE_SAVED_OBJ = True

# path to dataset
data_path = "/content/drive/MyDrive/dataset/Hamshahri-Corpus.txt"

if (not USE_SAVED_OBJ):
  # read the cotpus as a string
  corpus_raw = read_hamshahri_corpus(data_path)
  
  # extract the rarewords
  rare_words = get_rare_words(corpus_raw, 1)
  
  # split the input corpus into sentences separated by '.'
  sentences = corpus_raw.split('.')

  tokenized_sentences = []
  
  # clean each sentence and split it into tokens
  for sent in sentences:
    tokens = clean_sentence(sent)
    tokenized_sentences.append(tokens)
  
  # save the cleaned sentences in pickle file
  file = open("/content/drive/MyDrive/data/corpus.obj","wb")
  pickle.dump(tokenized_sentences,file)
  file.close()

else:
  
  # load the cleaned sentences by pickle from file
  file = open("/content/drive/MyDrive/data/corpus.obj","rb")
  tokenized_sentences = pickle.load(file)
  file.close()



In [ ]:
len(tokenized_sentences)

5984

In [ ]:
# Extract sequences of N-grams from corpus

# count of given words to predict the next word
window_size = 3

# length of sequence or n-gram
seq_length = window_size + 1

# store the n-grams in sequences list
sequences = list()

# extract a sentence of each 'seq_length' consecutive words from given sentences.
for tokens in tokenized_sentences:
  for i in range(seq_length, len(tokens)):
    seq = tokens[i-seq_length:i]
    line = ' '.join(seq)
    sequences.append(line)

print('Total Sequences: %d' % len(sequences))

Total Sequences: 130999


In [ ]:
sequences[5]

'ميكروبي عراق توسط كارشناسان'

In [ ]:

from keras.preprocessing.text import Tokenizer
# create the tokenizer and fit it on the input text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sequences)

# tokenize the sequences into encoded numbers
sequences = tokenizer.texts_to_sequences(sequences)


In [ ]:
sequences[5]

[4422, 226, 151, 346]

In [ ]:
# vocabulary size
vocab_size = len(tokenizer.word_index) + 1

In [ ]:
vocab_size

6213

In [ ]:
sequences = array(sequences)

# split the sequence of N-grams into input (x) and output (y)
X, y = sequences[:,:-1], sequences[:,-1]


In [ ]:
from tensorflow.keras.utils import to_categorical
# convert the output into categorical format required for training the model
y = to_categorical(y, num_classes=vocab_size)


In [ ]:
np.shape(y[0])

(6213,)

In [ ]:
y[0]

array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)

In [ ]:
np.shape(X)

(130999, 3)

In [ ]:
# split data into train, test, and validation sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.40, random_state=42)

length = int(len(X_test)/2)

y_validation = y_test[0:length]
X_validation = X_test[0:length]

y_test = y_test[length:2*length]
X_test = X_test[length:2*length]


In [ ]:
# The PlotLosses function, plots the validation and training loss function during the
#  training to give an insight of training the model
import keras
from matplotlib import pyplot as plt
from IPython.display import clear_output

class PlotLosses(keras.callbacks.Callback):
  
    def on_train_begin(self, logs={}):
        self.i = 0
        self.x = []
        self.losses = []
        self.val_losses = []
        
        self.fig = plt.figure()
        
        self.logs = []

    def on_epoch_end(self, epoch, logs={}):
        
        self.logs.append(logs)
        self.x.append(self.i)
        self.losses.append(logs.get('loss'))
        self.val_losses.append(logs.get('val_loss'))
        self.i += 1
        
        clear_output(wait=True)
        plt.plot(self.x, self.losses, label="loss")
        plt.plot(self.x, self.val_losses, label="val_loss")
        plt.legend()
        plt.show();
        
plot_losses = PlotLosses()

In [ ]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Embedding
from keras.layers import Flatten

# define model
model = Sequential()
model.add(Embedding(vocab_size, 50, input_length=window_size, name ='Embedding-layer'))
model.add(Flatten())
model.add(Dense(int(vocab_size/2), activation='relu', name='hidden-layer'))
model.add(Dense(vocab_size, activation='softmax', name='output-layer'))

# print summary of model's structure:
print(model.summary())

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
Embedding-layer (Embedding)  (None, 3, 50)             310650    
_________________________________________________________________
module_wrapper (ModuleWrappe (None, 150)               0         
_________________________________________________________________
hidden-layer (Dense)         (None, 3106)              469006    
_________________________________________________________________
output-layer (Dense)         (None, 6213)              19303791  
Total params: 20,083,447
Trainable params: 20,083,447
Non-trainable params: 0
_________________________________________________________________
None


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

# define checkpoint
checkpoint = ModelCheckpoint('/content/drive/MyDrive/models/model-{epoch:03d}--{val_accuracy:03f}.h5', verbose=1, monitor='val_loss',save_best_only=True, 
                             save_weights_only=True, mode='min')  

# compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# train the model
model.fit(X_train, y_train, 
          batch_size=50, 
          epochs=7, 
          validation_data=(X_validation, y_validation),
          callbacks=[checkpoint, plot_losses])



Epoch 1/7
  79/1572 [>.............................] - ETA: 5:46 - loss: 7.7572 - accuracy: 0.0400

KeyboardInterrupt: ignored

<Figure size 432x288 with 0 Axes>

### An example of Validation and Train loss per epoch:

![](https://drive.google.com/uc?export=view&id=1LkihDu-MFA6rpJq1TejJZd8ren4NzgcW)



In [ ]:

# load the best model
best_model_path = '/content/drive/MyDrive/models/model-002--0.112863.h5'
model.load_weights(best_model_path)

In [ ]:
# predict the class of test data
res = model.predict_classes(X_test[0:200])

/usr/local/lib/python3.7/dist-packages/tensorflow/python/keras/engine/sequential.py:455: UserWarning: `model.predict_classes()` is deprecated and will be removed after 2021-01-01. Please use instead:* `np.argmax(model.predict(x), axis=-1)`,   if your model does multi-class classification   (e.g. if it uses a `softmax` last-layer activation).* `(model.predict(x) > 0.5).astype("int32")`,   if your model does binary classification   (e.g. if it uses a `sigmoid` last-layer activation).
  warnings.warn('`model.predict_classes()` is deprecated and '


In [ ]:
# given the ID of word returns its corresponding word
def convert_ID_to_word(ID):   
  for word, index in tokenizer.word_index.items():
    if index == ID:
      return word
  return 

In [ ]:
# given the input sentence as an array of word Ids, returns the string of sentence
def get_sentence_from_IDs(x):
  sent = ''
  for elem in x:
    sent+= convert_ID_to_word(elem) + ' '
  return sent

In [ ]:

# print the test data with prediction of model
arr = [0,6, 13, 14, 17, 18, 22, 35, 53, 73, 105, 112, 124, 128, 137, 145, 154, 161, 185]
for i in arr:
  print( get_sentence_from_IDs(X_test[i]) +' : '+convert_ID_to_word(res[i]))




به گزارش خبرگزاري  : فرانسه
شهر در كشورهاي  : اسلامي
سال پيش آغاز  : شده
در پاريس اعلام  : كرد
شش سال محاسبه  : مي
كه بخش كشاورزي  : را
كردند اخيرا دست  : مي
مقدس از روز  : گذشته
آن در اين  : زمينه
وابسته به شهرداري  : تهران
آن در شهر  : تهران
صنايع دستي به  : عنوان
متحد هواپيمايي جمهوري  : اسلامي
دومين پروژه اتوبوس  : هاي
نظام جمهوري اسلامي  : ايران
بار درسال رقابتهاي  : فوتبال
در نقاط مختلف  : ايران
از نواحي شهر  : تهران
هاشمي رفسنجاني رئيس  : جمهوري
